# 联合类型、交叉类型与字面量

学习目标：能组合数据约束并用字面量表达有限状态，选择合适的标注、satisfies 与 const 断言。

前置知识：TypeScript 对象类型、类型别名、接口和只读元组；JavaScript 条件分支。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/05-unions-and-literals/。

1. [main.ts](scripts/05-unions-and-literals/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/05-unions-and-literals/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/05-unions-and-literals/tsconfig.json)、[tsconfig.errors.json](scripts/05-unions-and-literals/tsconfig.errors.json)：分别明确正常与反例文件范围。



Step 1：检查正常项目的类型。

```bash
npm run check:05
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:05
```

Step 3：运行正常示例。

```bash
npm run run:05
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:05
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/05-unions-and-literals/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 联合类型与公共操作

联合类型允许值属于任一成员。对尚未收窄的联合，操作必须对每个可能成员都成立；这不是把所有成员的能力相加。字符串与数组都有 length，因而可以共同读取；仅字符串具有的大小写方法则需要先判断。

交叉类型表达“同时满足”而不是“二选一”。两者组合时先按需要明确括号，避免把元素的选择与整体的选择混淆。

```typescript
export {};
function measure(value: string | readonly string[]): number { return value.length; }
function normalize(value: string | number): string {
  return typeof value === "string" ? value.trim() : value.toFixed(0);
}
console.log(measure(["甲", "乙"]), normalize(" TS "), normalize(3)); // 2 TS 3
```

以下片段来自独立的 type-errors.ts：

```typescript
function wrongUnion(value: string | number): string {
  return value.toUpperCase(); // number 分支没有此方法。
}
```

## 2 交叉类型及属性冲突

把 HasTitle & Timed 作为类型，就要求一个值同时具有两部分结构。交叉不会在运行时合并对象，也不是用右边属性覆盖左边。

如果同名必需属性分别要求 string 和 number，该属性需同时满足两者，结果为 never。接口 extends 通常直接在不兼容扩展处报错；交叉类型可以先形成，直到构造或使用不可能的成员时才暴露问题。

```typescript
type HasTitle = { title: string };
type Timed = { minutes: number };
type Lesson = HasTitle & Timed;
const lesson: Lesson = { title: "联合", minutes: 15 };
console.log(lesson.title, lesson.minutes); // 联合 15
```

以下片段来自独立的 type-errors.ts：

```typescript
type Conflicting = { id: string } & { id: number };
const conflict: Conflicting = { id: "x" }; // id 需要满足 string & number，即 never。
```

## 3 字面量表示有限值与状态

字符串和数值字面量类型把允许值缩小为列出的常量。State 只接受三种状态，Level 只接受三个等级；它们不在运行时自动验证输入。

若状态关联不同数据，用具有共同字面量字段的对象联合，让成功分支要求结果、失败分支要求原因。这样不必靠多个互相独立的可选字段猜测关系；下一章系统讨论这种可辨识联合的收窄。

```typescript
type State = "idle" | "ready" | "failed";
type Level = 1 | 2 | 3;
function badge(state: State, level: Level): string { return state + ":" + level; }
type Result = { state: "ready"; total: number } | { state: "failed"; reason: string };
const result: Result = { state: "ready", total: 2 };
console.log(badge("idle", 1), result.total); // idle:1 2
```

以下片段来自独立的 type-errors.ts：

```typescript
type KnownState = "idle" | "ready" | "failed";
const typo: KnownState = "read"; // 不属于列出的字面量集合。
type StateResult = { state: "ready"; total: number } | { state: "failed"; reason: string };
const missingData: StateResult = { state: "ready" }; // ready 分支必须包含 total。
```

## 4 字面量拓宽与 as const

字面量类型拓宽（literal widening）让可修改位置获得更一般的类型。let mode = "ready" 通常推断为 string；const 固定变量绑定，但对象的可写属性仍可能拓宽为 string。

as const 作用于字面量表达式：保留字面量类型、把对象字面量属性标为 readonly、把数组字面量推断为只读元组。它不会冻结运行时对象，也不会把引用进来的既有可写对象递归变为只读。

```typescript
let mode = "ready"; // string，可以稍后保存其他字符串。
const binding = "ready"; // 单一字面量类型。
const ordinary = { state: "ready" }; // state 为 string。
const tags = ["基础"];
const fixed = { state: "ready", position: [1, 2], tags } as const;
tags.push("实践");
const known: "ready" = fixed.state;
console.log(mode, binding, ordinary.state, known, fixed.tags.length); // ready ready ready ready 2
```

以下片段来自独立的 type-errors.ts：

```typescript
const widened = { state: "ready" };
const narrow: "ready" = widened.state; // 可写对象属性已拓宽为 string。
const constant = { state: "ready" } as const;
constant.state = "ready"; // 即使新值相同，也不能通过只读属性赋值。
```

## 5 标注、断言与 satisfies 的差别

类型标注给变量指定契约，后续读取按该契约检查。satisfies 从 TypeScript 4.9 起可验证表达式是否满足目标类型，同时保留比整个目标标注更具体的结果信息。目标也可提供上下文类型，不应将其概括成完全不影响推断。

下面字典中的 text 可以是字符串或二位数值元组。标注后的属性只能按联合读取；satisfies 版本仍可直接使用字符串方法。as 普通类型断言不承担同样的赋值验证，可能遮住必要成员缺失；它与 satisfies 都不会增加运行时校验。需要同时保留常量和检查结构时，可组合 as const satisfies。

```typescript
type Label = { text: string | [number, number] };
const annotated: Label = { text: "TS" };
const checked = { text: "TS" } satisfies Label;
const asserted = {} as Label; // 演示不可靠断言；不读取缺失成员的方法。
const setting = { state: "ready", level: 2 } as const satisfies { state: State; level: Level };
const upper = typeof annotated.text === "string" ? annotated.text.toUpperCase() : "坐标";
console.log(upper, checked.text.toUpperCase(), "text" in asserted, setting.level); // TS TS false 2
```

以下片段来自独立的 type-errors.ts：

```typescript
type LabelContract = { text: string | [number, number] };
const invalid = { text: true } satisfies LabelContract; // 不兼容值被拒绝。
const omitted = {} satisfies LabelContract; // 缺少 text 被拒绝。
const annotatedLabel: LabelContract = { text: "TS" };
annotatedLabel.text.toUpperCase(); // 标注把读取类型设为联合，需要先收窄。
```

## 本章小结

联合表示可选值集合，交叉要求同时满足约束，冲突属性不会被覆盖。字面量能表达有限状态；as const 保留常量信息，satisfies 检查结构，普通断言不提供运行时依据。

## 练习

1. 为上传任务建立 idle、done、failed 联合，done 必有文件名、failed 必有原因。正常数据应检查通过，缺少对应数据的对象应失败。
2. 把含有数值和字符串的配置写成 satisfies 形式，数值属性能直接调用 toFixed，字符串属性能直接调用 toUpperCase；改成统一联合标注后说明为何需要收窄。
3. 构造两个同名属性冲突的交叉类型，定位 never 属性；修正领域模型后正常赋值，不使用 as。

### 提示

1. 用 state 作为共同字面量字段，每个联合成员分别写出需要的数据。
2. 用 { count: 2, title: "TS" } satisfies { [key: string]: string | number }，再与相同初值的显式标注比较。
3. 比较同一个属性在两侧各自要求什么，不要按对象展开的覆盖规则理解 &。


### 参考解析

1. 可用 { state: "idle" } | { state: "done"; filename: string } | { state: "failed"; reason: string }；done 缺 filename、failed 缺 reason 都应诊断失败。
2. satisfies 后 count 保留 number、title 保留 string；若变量直接标注为 { [key: string]: string | number }，读取相应键得到联合，调用特有方法前需缩窄。
3. 例如 string 与 number 的 id 交叉为 never；若领域要求一种统一的标识，统一两侧 id 类型后再构造值，而不是断言跳过矛盾。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Everyday Types：联合、字面量与 Literal Inference](https://www.typescriptlang.org/docs/handbook/2/everyday-types.html#union-types)；[Object Types：Intersection Types 与冲突](https://www.typescriptlang.org/docs/handbook/2/objects.html#intersection-types)；[3.4：const assertions 与限制](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-4.html#const-assertions)；[4.9：satisfies](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-9.html#the-satisfies-operator)。 |
| npm 官方文档 | [npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。 |
